<div dir="rtl">

# 04 — اختيار الأدلة: عشوائي مقابل greedy_entropy مقابل expected_information_gain

ثلاث استراتيجيات لاختيار k دليل إضافي (بعد `INITIAL_EVIDENCE`) من الأدلة الفعلية لكل مريض، مقارنةً على نفس
العيّنة الفرعية الطبقية من `validate` (10,000 صف، `random_state=42`) ونفس النموذج المدرَّب
(`class_weight="balanced"`, عيّنة تدريب 200,000، `random_state=42` — مطابق تماماً لـ`03_train.ipynb`):

1. **`random`** — إسقاط عشوائي (خط الأساس من `03_train.ipynb`).
2. **`greedy_entropy`** — جشع بأقل إنتروبيا فعلية (baked من التشغيل السابق لهذا النوتبوك — **نتيجة سلبية
   مهمة، محفوظة كاملة دون حذف**).
3. **`expected_information_gain`** (جديد) — جشع بأعلى **مكسب معلوماتي متوقَّع**: جدول `P(evidence|disease)`
   محسوب مرة واحدة من `train` الكامل كنموذج Bayes ساذج، والمعيار هو **التوقع على الإجابات الممكنة** (لا
   الجواب الفعلي كما في `greedy_entropy`). **التشخيص النهائي في كل الحالات يبقى بالنموذج المدرَّب
   (`LogisticRegression`)** — Bayes الساذج يُستخدَم فقط لاختيار الأدلة، لا للتنبؤ.

قبل الجدول النهائي: **تحقيق صريح** في سبب اختلاف أرقام `random` بين هذا النوتبوك (تشغيله الأول) و
`03_train.ipynb` — طُلب توثيقه قبل أي مقارنة أخرى.

</div>

In [1]:
import ast
import json
import sys
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

sys.path.insert(0, "..")
from src.selection import ecode, build_nb_presence_table, select_evidence_eig

RANDOM_STATE = 42
SAMPLE_SIZE = 200_000
SUBSAMPLE_SIZE = 10_000
K_VALUES = (3, 5, 8, 12)
K_MAX = 12
EPS = 1e-12
ALPHA = 1.0  # تنعيم لابلاس لجدول Bayes الساذج

DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")

<div dir="rtl">

## تحميل المصفوفات وإعادة بناء نفس نموذج `03_train.ipynb`

</div>

In [2]:
with open(DATA_PROCESSED / "feature_names.json", encoding="utf-8") as f:
    feature_names = json.load(f)
label_encoder = joblib.load(DATA_PROCESSED / "label_encoder.joblib")

X_train_full = sparse.load_npz(DATA_PROCESSED / "train_features.npz")
y_train_full = np.load(DATA_PROCESSED / "train_labels.npy")
X_validate = sparse.load_npz(DATA_PROCESSED / "validate_features.npz")
y_validate = np.load(DATA_PROCESSED / "validate_labels.npy")

assert X_train_full.shape[0] == 1_025_602
assert X_train_full.shape[1] == len(feature_names) == 974
assert X_validate.shape[1] == len(feature_names)

evidence_columns = feature_names[:-2]
item_to_col_idx = {name: idx for idx, name in enumerate(evidence_columns)}
N_EVID = len(evidence_columns)
print(f"X_train_full: {X_train_full.shape}, X_validate: {X_validate.shape}, أعمدة أدلة: {N_EVID}")

X_train_full: (1025602, 974), X_validate: (132448, 974), أعمدة أدلة: 972


In [3]:
with open(DATA_RAW / "release_conditions.json", encoding="utf-8") as f:
    conditions = json.load(f)
severity_by_class = np.array([conditions[c]["severity"] for c in label_encoder.classes_])
severity_1_classes = set(np.where(severity_by_class == 1)[0])
print(f"severity=1: {len(severity_1_classes)} من أصل {len(label_encoder.classes_)}")

severity=1: 5 من أصل 49


In [4]:
def top_k_accuracy(y_true, proba, k):
    top_k = np.argpartition(-proba, kth=k - 1, axis=1)[:, :k]
    return (top_k == y_true[:, None]).any(axis=1).mean()


def severity1_miss_rate(y_true, y_pred, sev1):
    mask = np.isin(y_true, list(sev1))
    n = int(mask.sum())
    missed = int(np.sum(y_pred[mask] != y_true[mask]))
    return (missed / n if n else float("nan")), n


def evaluate(model, X, y_true):
    proba = model.predict_proba(X)
    y_pred = model.predict(X)
    miss_rate, n_sev1 = severity1_miss_rate(y_true, y_pred, severity_1_classes)

    # severity معكوسة: رقم أكبر = أقل خطورة. undertriage = توقّع أقل خطورة من الحقيقة.
    sev_true = severity_by_class[y_true]
    sev_pred = severity_by_class[y_pred]
    undertriage_rate = float(np.mean(sev_pred > sev_true))
    same_severity_confusion = float(np.mean((y_pred != y_true) & (sev_pred == sev_true)))

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Top-3": top_k_accuracy(y_true, proba, 3),
        "Top-5": top_k_accuracy(y_true, proba, 5),
        "Macro-F1": f1_score(y_true, y_pred, average="macro"),
        "نسبة الخطأ severity=1": miss_rate,
        "عدد حالات severity=1": n_sev1,
        "undertriage_rate": undertriage_rate,
        "same_severity_confusion": same_severity_confusion,
    }


def encode_evidences(evidences_parsed, item_to_col_idx, n_evidence_cols):
    exploded = evidences_parsed.explode()
    col_idx = exploded.map(item_to_col_idx).to_numpy(dtype=np.int64)
    row_idx = exploded.index.to_numpy()
    data = np.ones(len(exploded), dtype=np.int8)
    return sparse.coo_matrix(
        (data, (row_idx, col_idx)), shape=(len(evidences_parsed), n_evidence_cols)
    ).tocsr()


# ecode() مستوردة من src/selection.py أعلاه — لا تُعرَّف هنا لتفادي التكرار


def build_X_from_kept(kept_items_list, age_arr, sex_arr):
    kept_series = pd.Series(kept_items_list)
    evid = encode_evidences(kept_series, item_to_col_idx, N_EVID)
    age_c = sparse.csr_matrix(np.asarray(age_arr, dtype=float).reshape(-1, 1))
    sex_c = sparse.csr_matrix(np.asarray(sex_arr, dtype=float).reshape(-1, 1))
    return sparse.hstack([evid, age_c, sex_c]).tocsr()

In [5]:
X_train, _, y_train, _ = train_test_split(
    X_train_full, y_train_full, train_size=SAMPLE_SIZE, stratify=y_train_full, random_state=RANDOM_STATE
)
classes = np.unique(y_train)
counts = np.array([(y_train == c).sum() for c in classes])
balanced_weights = len(y_train) / (len(classes) * counts)
class_weight = dict(zip(classes.tolist(), balanced_weights.tolist()))

t0 = time.time()
model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight=class_weight)
model.fit(X_train, y_train)
print(f"إعادة تدريب balanced: تم بعد {time.time() - t0:.1f}s (n_iter={model.n_iter_})")

إعادة تدريب balanced: تم بعد 280.7s (n_iter=[877])


<div dir="rtl">

## عيّنة `validate` الفرعية الطبقية (10,000 صف) وبنية الأدلة لكل مريض

</div>

In [6]:
df_val = pd.read_csv(DATA_RAW / "release_validate_patients.csv")
df_val["EVIDENCES_PARSED"] = df_val["EVIDENCES"].apply(ast.literal_eval)
codes_per_row_full = df_val["EVIDENCES_PARSED"].apply(lambda items: sorted(set(ecode(it) for it in items)))
initial_code_full = df_val["INITIAL_EVIDENCE"].apply(ecode)
items_full = df_val["EVIDENCES_PARSED"]

idx_all = np.arange(len(y_validate))
idx_sub, _ = train_test_split(idx_all, train_size=SUBSAMPLE_SIZE, stratify=y_validate, random_state=RANDOM_STATE)
idx_sub = np.sort(idx_sub)
print(f"عيّنة validate الفرعية: {len(idx_sub)} صف (من {len(idx_all)})")

عيّنة validate الفرعية: 10000 صف (من 132448)


In [7]:
y_sub = y_validate[idx_sub]
age_col_full = X_validate[:, N_EVID].toarray().ravel()
sex_col_full = X_validate[:, N_EVID + 1].toarray().ravel()
age_sub = age_col_full[idx_sub]
sex_sub = sex_col_full[idx_sub]

n_sub = len(idx_sub)
codes_per_row = [codes_per_row_full.iloc[i] for i in idx_sub]
initial_code = [initial_code_full.iloc[i] for i in idx_sub]

items_by_code = []
for i in idx_sub:
    d = {}
    for it in items_full.iloc[i]:
        d.setdefault(ecode(it), []).append(it)
    items_by_code.append(d)

bad = sum(1 for ic, codes in zip(initial_code, codes_per_row) if ic not in codes)
assert bad == 0, f"{bad} صفاً INITIAL_EVIDENCE فيها ليس ضمن EVIDENCES"

<div dir="rtl">

## تحقيق: لماذا اختلفت أرقام `random` عن `03_train.ipynb`؟

`03_train.ipynb` استخدم `rng = default_rng(42)` **واحداً يُستهلَك تسلسلياً عبر الـ132,448 صف كاملة** بترتيب
ملف `validate` الأصلي. النوتبوك الحالي (تشغيله الأول) استخدم نفس آلية `default_rng(42)` لكن **مُستهلَكاً فقط
عبر الـ10,000 صف الفرعية** (ترتيب مُرتَّب من مواضعها الأصلية). هذان تصميمان مختلفان لنفس البذرة، لهما **أثران
منفصلان** على الرقم النهائي:

1. **تباين المعاينة (sampling variance)**: حتى لو كانت اختيارات كل صف مطابقة تماماً، حساب الدقة على 10,000
   صف فقط بدل 132,448 يُعطي رقماً مختلفاً قليلاً بطبيعته (عيّنة أصغر = تذبذب إحصائي أكبر).
2. **أثر ترتيب استهلاك المولّد العشوائي (RNG consumption order)**: بما أن `rng` واحد يُستهلَك تسلسلياً، تغيير
   *مجموعة* الصفوف المُكرَّرة عليها يُغيّر *أي* رقم عشوائي يقع على كل صف — حتى الصفوف المشتركة بين الحالتين
   تحصل على اختيار عشوائي مختلف.

نتحقق من الأثرين بشكل منفصل: نُشغّل نفس آلية `03_train.ipynb` (rng عبر الـ132,448 كاملة) ثم نقيس الدقة مرتين:
مرة على كل الـ132,448 (يجب أن يطابق `03_train.ipynb` تماماً)، ومرة بتقييد النتيجة على صفوف العيّنة الفرعية
فقط (نفس الاختيارات، عيّنة أصغر) — هذا يعزل أثر (1). ثم نقارنها بتشغيل النوتبوك الأصلي (rng عبر الـ10,000 فقط)
— الفرق المتبقي هو أثر (2).

</div>

In [8]:
def make_random_ablated_seqrng(k, row_positions, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    kept = {}
    for i in row_positions:
        codes = codes_per_row_full.iloc[i]
        ic = initial_code_full.iloc[i]
        remaining = [c for c in codes if c != ic]
        n_take = min(k, len(remaining))
        sampled = (
            rng.choice(np.array(remaining, dtype=object), size=n_take, replace=False).tolist()
            if n_take > 0
            else []
        )
        keep_codes = set(sampled) | {ic}
        items = items_full.iloc[i]
        kept[i] = [it for it in items if ecode(it) in keep_codes]
    return kept


diag_rows = []
for k in K_VALUES:
    kept_full = make_random_ablated_seqrng(k, range(len(y_validate)))  # == 03_train.ipynb بالضبط

    kept_list_full_ordered = [kept_full[i] for i in range(len(y_validate))]
    X_full = build_X_from_kept(kept_list_full_ordered, age_col_full, sex_col_full)
    metrics_03 = evaluate(model, X_full, y_validate)

    kept_sub_from_full = [kept_full[i] for i in idx_sub]  # نفس الاختيارات، عيّنة 10 آلاف فقط
    X_sub_from_full = build_X_from_kept(kept_sub_from_full, age_sub, sex_sub)
    metrics_A = evaluate(model, X_sub_from_full, y_sub)

    kept_B = make_random_ablated_seqrng(k, idx_sub.tolist())  # rng عبر الـ10 آلاف فقط (تشغيل 04 الأصلي)
    kept_list_B = [kept_B[i] for i in idx_sub]
    X_B = build_X_from_kept(kept_list_B, age_sub, sex_sub)
    metrics_B = evaluate(model, X_B, y_sub)

    diag_rows.append(
        {
            "k": k,
            "03 (132,448 صف)": metrics_03["Accuracy"],
            "نفس الاختيارات / 10,000 فقط": metrics_A["Accuracy"],
            "04 الأصلي (rng على 10,000)": metrics_B["Accuracy"],
            "أثر تباين المعاينة": metrics_A["Accuracy"] - metrics_03["Accuracy"],
            "أثر ترتيب RNG": metrics_B["Accuracy"] - metrics_A["Accuracy"],
        }
    )
    print(
        f"k={k}: 03(132k)={metrics_03['Accuracy']:.6f}  "
        f"نفس_الاختيارات/10k={metrics_A['Accuracy']:.6f}  "
        f"04_الأصلي(rng/10k)={metrics_B['Accuracy']:.6f}"
    )

diag_df = pd.DataFrame(diag_rows).set_index("k")

k=3: 03(132k)=0.569333  نفس_الاختيارات/10k=0.564600  04_الأصلي(rng/10k)=0.571200
k=5: 03(132k)=0.785508  نفس_الاختيارات/10k=0.786200  04_الأصلي(rng/10k)=0.787900
k=8: 03(132k)=0.927300  نفس_الاختيارات/10k=0.924900  04_الأصلي(rng/10k)=0.927300
k=12: 03(132k)=0.980234  نفس_الاختيارات/10k=0.978300  04_الأصلي(rng/10k)=0.981300


In [9]:
diag_df

    03 (132,448 صف)  نفس الاختيارات / 10,000 فقط  04 الأصلي (rng على 10,000)  أثر تباين المعاينة  أثر ترتيب RNG
k
3           0.569333                        0.5646                      0.5712            -0.004733          0.0066
5           0.785508                        0.7862                      0.7879             0.000692          0.0017
8           0.927300                        0.9249                      0.9273            -0.002400          0.0024
12          0.980234                        0.9783                      0.9813            -0.001934          0.0030

<div dir="rtl">

**النتيجة**: الأثران معاً صغيران (≤ 0.7 نقطة مئوية لكل منهما، عبر كل قيم k)، ومتوافقان تماماً مع تذبذب أخذ
عيّنات عادي — **ليس خطأً في الكود**. أثر ترتيب RNG موجب باستمرار (+0.17 إلى +0.66 نقطة) وأثر تباين المعاينة
متذبذب الإشارة (كما يُتوقَّع من ضجيج إحصائي حقيقي لا انحياز منهجي). **الإصلاح المُتَّبع أدناه**: بذرة عشوائية
**مستقلة لكل صف** (`default_rng((RANDOM_STATE, absolute_row_position))`) بدل بذرة واحدة تُستهلَك تسلسلياً —
هذا يجعل اختيار كل صف محدَّداً بمعزل تام عن حجم/ترتيب العيّنة المحيطة به، فلا يتكرر هذا الالتباس مستقبلاً.

</div>

<div dir="rtl">

## `random` — عشوائي ببذرة مستقلة لكل صف

</div>

In [10]:
def make_random_ablated_indep(k):
    kept_items_list = []
    for local_i, abs_i in enumerate(idx_sub):
        rng = np.random.default_rng((RANDOM_STATE, int(abs_i)))
        codes = codes_per_row[local_i]
        ic = initial_code[local_i]
        remaining = [c for c in codes if c != ic]
        n_take = min(k, len(remaining))
        sampled = (
            rng.choice(np.array(remaining, dtype=object), size=n_take, replace=False).tolist()
            if n_take > 0
            else []
        )
        keep_codes = set(sampled) | {ic}
        kept_items_list.append([it for c in keep_codes for it in items_by_code[local_i][c]])
    return kept_items_list


random_results = []
for k in K_VALUES:
    kept = make_random_ablated_indep(k)
    X_k = build_X_from_kept(kept, age_sub, sex_sub)
    metrics = evaluate(model, X_k, y_sub)
    avg_kept = float(np.mean([len(x) for x in kept]))
    random_results.append({"k": k, "الطريقة": "random", "متوسط الأدلة": avg_kept, **metrics})
    print(f"[random] k={k}: Accuracy={metrics['Accuracy']:.4f}")

[random] k=3: Accuracy=0.5687
[random] k=5: Accuracy=0.7872
[random] k=8: Accuracy=0.9241
[random] k=12: Accuracy=0.9810


<div dir="rtl">

## `greedy_entropy` — جشع بأقل إنتروبيا فعلية (محفوظ كما هو — نتيجة سلبية مهمة)

**لا حذف**: هذه هي نفس النتيجة المُسلَّمة سابقاً (نفس الكود، نفس البذرة، نفس النموذج → نتيجة حتمية مطابقة).
في كل خطوة، لكل مرشَّح من الأدلة المتبقية الفعلية، نبني الترميز (المُبقى + هذا المرشَّح) بقيمته **الحقيقية**
ونحسب إنتروبيا `predict_proba` الناتجة، ونختار الأقل إنتروبيا. راجع التفسير النقدي في الأسفل — هذه الطريقة
تراجعت بشدة أمام `random` وأمام `expected_information_gain` أدناه، خصوصاً على severity=1.

</div>

In [11]:
kept_items_state = [items_by_code[i][initial_code[i]][:] for i in range(n_sub)]
remaining_state_ge = [[c for c in codes_per_row[i] if c != initial_code[i]] for i in range(n_sub)]
greedy_checkpoints = {}
t_ge0 = time.time()

for step in range(1, K_MAX + 1):
    active = [i for i in range(n_sub) if len(remaining_state_ge[i]) > 0]
    if active:
        task_lists, task_row, task_code, row_offsets = [], [], [], {}
        pos = 0
        for i in active:
            cands = remaining_state_ge[i]
            row_offsets[i] = (pos, pos + len(cands))
            pos += len(cands)
            for c in cands:
                task_lists.append(kept_items_state[i] + items_by_code[i][c])
                task_row.append(i)
                task_code.append(c)
        age_tasks = np.array([age_sub[r] for r in task_row])
        sex_tasks = np.array([sex_sub[r] for r in task_row])
        X_tasks = build_X_from_kept(task_lists, age_tasks, sex_tasks)
        proba_tasks = model.predict_proba(X_tasks)
        entropy_tasks = -np.sum(proba_tasks * np.log(proba_tasks + EPS), axis=1)
        for i in active:
            start, end = row_offsets[i]
            local_best = start + int(np.argmin(entropy_tasks[start:end]))
            chosen_code = task_code[local_best]
            kept_items_state[i] = kept_items_state[i] + items_by_code[i][chosen_code]
            remaining_state_ge[i].remove(chosen_code)
    if step in K_VALUES:
        greedy_checkpoints[step] = [list(x) for x in kept_items_state]

print(f"إجمالي وقت greedy_entropy: {time.time() - t_ge0:.1f}s")

greedy_entropy_results = []
for k in K_VALUES:
    kept = greedy_checkpoints[k]
    X_k = build_X_from_kept(kept, age_sub, sex_sub)
    metrics = evaluate(model, X_k, y_sub)
    avg_kept = float(np.mean([len(x) for x in kept]))
    greedy_entropy_results.append({"k": k, "الطريقة": "greedy_entropy", "متوسط الأدلة": avg_kept, **metrics})
    print(f"[greedy_entropy] k={k}: Accuracy={metrics['Accuracy']:.4f}")

إجمالي وقت greedy_entropy: 5.1s
[greedy_entropy] k=3: Accuracy=0.6746
[greedy_entropy] k=5: Accuracy=0.7695
[greedy_entropy] k=8: Accuracy=0.8711
[greedy_entropy] k=12: Accuracy=0.9316


<div dir="rtl">

## `expected_information_gain` — جشع بأعلى مكسب معلوماتي متوقَّع (Bayes ساذج)

**جدول `P(code present | disease)` من `train` الكامل (1,025,602 صف) مرة واحدة**: لكل من الأكواد الـ223
المميزة، وكل مرض من الـ49، نسبة مرضى ذلك المرض الذين ظهر الكود لديهم فعلاً (تنعيم لابلاس `α=1`) — محسوبة
مباشرة من `X_train_full` المُرمَّز (لا حاجة لإعادة تحليل CSV الخام). **مُبسَّط عمداً**: نموذج Bayes الساذج
يعامل كل كود كمتغيّر ثنائي (موجود/غائب) بغض النظر عن كونه B أو C أو M — لا يُنمذِج *أي* قيمة تحديداً لأدلة
C/M متعددة القيم، فقط "هل ظهر الكود أصلاً" — تبسيط ضروري للتعامل مع الأدلة متعددة القيم (بعضها بعشرات القيم
الممكنة) دون فضاء نتائج هائل.

**خوارزمية الاختيار (لكل مريض، لكل خطوة)**:
1. نحافظ على معتقد لاحق `P(disease | الأدلة المكشوفة حتى الآن)` عبر Bayes، مبتدئين من الأولوية القبلية
   (تكرار كل مرض في `train`) مُحدَّثة بـ`INITIAL_EVIDENCE`.
2. لكل كود مرشَّح **من الأدلة الفعلية المتبقية لدى المريض** (نفس مجموعة المرشَّحين المُستخدَمة في
   `greedy_entropy`/`random`، لضمان مقارنة عادلة): نحسب **التوقع** على النتيجتين الممكنتين (ظهور/غياب الكود)
   — لا القيمة الفعلية — للمعتقد اللاحق الناتج، ونحسب إنتروبيا كل منهما، ونأخذ المتوسط المرجَّح باحتمالهما.
3. نختار الكود صاحب أقل إنتروبيا **متوقَّعة**. **بعدها فقط** نكشف قيمته **الحقيقية** (دائماً "موجود"، لأن
   المرشَّحين من أدلة المريض الفعلية) ونحدِّث المعتقد اللاحق بها فعلياً، ونكرر.

الفرق الجوهري عن `greedy_entropy`: هناك اخترنا بناءً على الإنتروبيا **الفعلية بعد الكشف** (نموذج قد "يغش"
بالنظر للإجابة الحقيقية قبل اختيار السؤال)؛ هنا الاختيار **لا يرى الإجابة الحقيقية إطلاقاً** — فقط التوزيع
الاحتمالي لما قد تكون عليه، وهذا هو التعريف الصحيح لمكسب معلوماتي متوقَّع.

**`build_nb_presence_table` و`select_evidence_eig` مستوردتان من `src/selection.py`** — لا تكرار لهذا المنطق
هنا؛ نفس الدالتين تُستخدَمان في `06_demo.ipynb` (مصدر حقيقة واحد لخوارزمية الاختيار). الاختيار تراكمي: أول k
كود من `code_sequence` هي نفسها أول k كود لأي k_max أكبر، فنقاط التفتيش (k=3,5,8,12) تُبنى من بادئات قائمة
واحدة محسوبة مرة واحدة، لا بإعادة التشغيل لكل k.

</div>

In [12]:
t0 = time.time()
P_present, class_prior, all_codes, code_to_idx = build_nb_presence_table(
    X_train_full, y_train_full, evidence_columns, len(label_encoder.classes_), alpha=ALPHA
)
print(f"عدد الأكواد المميزة: {len(all_codes)}")
print(f"بناء الجدول: تم بعد {time.time() - t0:.1f}s | شكل الجدول: {P_present.shape}")

عدد الأكواد المميزة: 223
بناء الجدول: تم بعد 2.3s | شكل الجدول: (49, 223)


In [13]:
t_eig0 = time.time()
code_sequence = select_evidence_eig(codes_per_row, initial_code, P_present, class_prior, code_to_idx, K_MAX)
print(f"إجمالي وقت اختيار expected_information_gain (12 خطوة، كل الصفوف): {time.time() - t_eig0:.1f}s")

# إعادة بناء نقاط التفتيش من بادئات code_sequence — الاختيار تراكمي (راجع src/selection.py)
eig_checkpoints = {}
for k in K_VALUES:
    eig_checkpoints[k] = [
        items_by_code[i][initial_code[i]][:] + [it for c in code_sequence[i][:k] for it in items_by_code[i][c]]
        for i in range(n_sub)
    ]
    print(f"k={k}: أول صف — الأكواد المُختارة: {code_sequence[0][:k]}")

إجمالي وقت اختيار expected_information_gain (12 خطوة، كل الصفوف): 6.4s
k=3: أول صف — الأكواد المُختارة: ['E_134', 'E_201', 'E_88']
k=5: أول صف — الأكواد المُختارة: ['E_134', 'E_201', 'E_88', 'E_227', 'E_79']
k=8: أول صف — الأكواد المُختارة: ['E_134', 'E_201', 'E_88', 'E_227', 'E_79', 'E_129', 'E_181', 'E_53']
k=12: أول صف — الأكواد المُختارة: ['E_134', 'E_201', 'E_88', 'E_227', 'E_79', 'E_129', 'E_181', 'E_53', 'E_91', 'E_161', 'E_94', 'E_130']


In [14]:
eig_results = []
for k in K_VALUES:
    kept = eig_checkpoints[k]
    X_k = build_X_from_kept(kept, age_sub, sex_sub)
    metrics = evaluate(model, X_k, y_sub)
    avg_kept = float(np.mean([len(x) for x in kept]))
    eig_results.append({"k": k, "الطريقة": "expected_information_gain", "متوسط الأدلة": avg_kept, **metrics})
    print(f"[EIG] k={k}: Accuracy={metrics['Accuracy']:.4f}")

[EIG] k=3: Accuracy=0.7397
[EIG] k=5: Accuracy=0.8943
[EIG] k=8: Accuracy=0.9411
[EIG] k=12: Accuracy=0.9792


<div dir="rtl">

## الجدول الثلاثي النهائي: random / greedy_entropy / expected_information_gain

</div>

In [15]:
combined = pd.DataFrame(random_results + greedy_entropy_results + eig_results)
pivot = combined.pivot(
    index="k", columns="الطريقة", values=["متوسط الأدلة", "Accuracy", "Top-3", "Macro-F1", "نسبة الخطأ severity=1"]
)
pivot = pivot.reindex(columns=["random", "greedy_entropy", "expected_information_gain"], level=1)
pivot

        متوسط الأدلة                                          Accuracy                                            Top-3                                           Macro-F1                                          نسبة الخطأ severity=1
الطريقة       random greedy_entropy expected_information_gain   random greedy_entropy expected_information_gain  random greedy_entropy expected_information_gain    random greedy_entropy expected_information_gain                random greedy_entropy expected_information_gain
k
3             4.8682         6.8034                    4.5925   0.5687         0.6746                    0.7397  0.7462         0.7975                    0.8852  0.554880       0.637966                  0.680824              0.603196       0.507324                  0.207723
5             7.4383         9.2785                    7.0150   0.7872         0.7695                    0.8943  0.9162         0.8907                    0.9702  0.742305       0.714580                  0.843003   

<div dir="rtl">

## `undertriage_rate` و`same_severity_confusion` — نفس الجدول بمقياسين إضافيين

- **`undertriage_rate`** = نسبة الحالات `severity[y_pred] > severity[y_true]`. **تنويه**: الترقيم معكوس —
  `severity=1` الأشد خطورة، `severity=5` الأخف. `severity[pred] > severity[true]` يعني **توقّع درجة خطورة
  أقل من الحقيقة** — تحت-الفرز الخطر فعلياً (مثال: NSTEMI/STEMI حقيقي `severity=1` يُصنَّف رجفاناً أذينياً
  `severity=3`).
- **`same_severity_confusion`** = خطأ تشخيصي (`y_pred != y_true`) لكن **بنفس درجة الخطورة** — التباس ضمن نفس
  فئة الإلحاح (مثال: Anaphylaxis وLarygospasm، كلاهما `severity=1`)، وليس تراجعاً في تقدير الخطورة.

هذان المقياسان **لا يحذفان** "نسبة الخطأ severity=1" أعلاه — تلك تحسب أي خطأ ضمن أمراض severity=1 الحقيقية
تحديداً (undertriage + same_severity_confusion معاً، لكن مُقيَّدة بتلك الفئة فقط).

</div>

In [16]:
pivot_undertriage = combined.pivot(
    index="k", columns="الطريقة", values=["undertriage_rate", "same_severity_confusion"]
)
pivot_undertriage = pivot_undertriage.reindex(columns=["random", "greedy_entropy", "expected_information_gain"], level=1)
pivot_undertriage

        undertriage_rate                                          same_severity_confusion
الطريقة            random greedy_entropy expected_information_gain           random greedy_entropy expected_information_gain
k
3                   0.1121         0.0780                    0.0590           0.1265         0.1150                    0.0800
5                   0.0639         0.0613                    0.0130           0.0685         0.0902                    0.0510
8                   0.0251         0.0381                    0.0029           0.0278         0.0558                    0.0348
12                  0.0066         0.0244                    0.0034           0.0063         0.0370                    0.0150

<div dir="rtl">

## سؤال: هل يختفي شذوذ k=8 (ارتفاع "نسبة الخطأ severity=1" لـ`expected_information_gain`) تحت `undertriage_rate`؟

**نعم — يختفي تماماً.** `undertriage_rate` لـ`expected_information_gain` **ينخفض رتيباً**:
`k=3: 5.90% → k=5: 1.30% → k=8: 0.29% → k=12: 0.34%` (الفرق بين k=8 وk=12 ضئيل جداً وداخل حدود الضجيج، وليس
ارتفاعاً حقيقياً بحجم ما ظهر في "نسبة الخطأ severity=1": `5.1% → 9.6%`).

**السبب**: "نسبة الخطأ severity=1" (من `03_train.ipynb`) تحسب *أي* خطأ ضمن حالات severity=1 الحقيقية — سواء
كان تراجعاً حقيقياً في الخطورة (`undertriage`) أو مجرد التباس مع مرض آخر **بنفس** الخطورة
(`same_severity_confusion`). تحليل مجموعة التراجع (k=5→k=8) في القسم التالي يُثبت أن الشذوذ كان **95.7%
منه (110 من 115 حالة) التباساً بين Anaphylaxis وLarygospasm — كلاهما severity=1** — لا تراجعاً في تقدير
الخطورة. من ناحية `undertriage_rate` تحديداً، لا فرق: كلا التشخيصين (الصحيح والخطأ) بنفس درجة الخطورة القصوى،
فلا يُسجَّل أي "تحت-فرز" في هذه الحالات إطلاقاً — وهذا يفسّر تماماً لماذا يختفي الشذوذ تحت هذا المقياس تحديداً.

**الخلاصة**: الشذوذ في "نسبة الخطأ severity=1" عند k=8 كان **قلقاً تشخيصياً حقيقياً** (تمييز خاطئ بين حالتين
طارئتين متشابهتين) **وليس قلقاً في تصنيف الخطورة نفسها** — تمييز مهم عند تقييم أثره العملي على منطق الفرز.

</div>

<div dir="rtl">

## تحليل مجموعة التراجع: حالات severity=1 صحّت عند k=5 وأخطأت عند k=8 (`expected_information_gain`)

**عيّنة تحقق أكبر (30,000 صف، `random_state=42`)**: قسم منفصل بعيّنة أكبر من العيّنة الرئيسية (10,000) —
النمط غير الرتيب في "نسبة الخطأ severity=1" تأكَّد بنفس الحجم تقريباً عند مضاعفة العيّنة 3 مرات (751→2,254
حالة severity=1)، فهذا التحليل يستخدم نفس العيّنة الأكبر لضمان قاعدة إحصائية أوسع لتحليل مجموعة تراجع صغيرة
العدد أصلاً.

</div>

In [17]:
from collections import Counter

SUBSAMPLE_SIZE_VERIFY = 30_000
idx_verify, _ = train_test_split(
    idx_all, train_size=SUBSAMPLE_SIZE_VERIFY, stratify=y_validate, random_state=RANDOM_STATE
)
idx_verify = np.sort(idx_verify)
y_verify = y_validate[idx_verify]
age_verify = age_col_full[idx_verify]
sex_verify = sex_col_full[idx_verify]
n_verify = len(idx_verify)

codes_per_row_v = [codes_per_row_full.iloc[i] for i in idx_verify]
initial_code_v = [initial_code_full.iloc[i] for i in idx_verify]
items_by_code_v = []
for i in idx_verify:
    d = {}
    for it in items_full.iloc[i]:
        d.setdefault(ecode(it), []).append(it)
    items_by_code_v.append(d)

n_sev1_verify = int(np.isin(y_verify, list(severity_1_classes)).sum())
print(f"عدد حالات severity=1 في عيّنة التحقق: {n_sev1_verify} من أصل {n_verify}")

عدد حالات severity=1 في عيّنة التحقق: 2254 من أصل 30000


In [18]:
# إعادة تشغيل EIG على عيّنة التحقق مع تتبّع تسلسل الأكواد المُختارة صراحةً
initial_code_idx_v = [code_to_idx[ic] for ic in initial_code_v]
posterior_v = np.tile(class_prior, (n_verify, 1))
for i in range(n_verify):
    posterior_v[i] = posterior_v[i] * P_present[:, initial_code_idx_v[i]]
    posterior_v[i] /= posterior_v[i].sum()

kept_state_v = [items_by_code_v[i][initial_code_v[i]][:] for i in range(n_verify)]
remaining_v = [[c for c in codes_per_row_v[i] if c != initial_code_v[i]] for i in range(n_verify)]
code_sequence_v = [[] for _ in range(n_verify)]
checkpoints_v = {}

for step in range(1, K_MAX + 1):
    for i in range(n_verify):
        cands = remaining_v[i]
        if not cands:
            continue
        cand_idx = np.array([code_to_idx[c] for c in cands])
        post_i = posterior_v[i]
        p_col = P_present[:, cand_idx]
        p_present_marg = post_i @ p_col
        post_if_present = post_i[:, None] * p_col
        post_if_present = post_if_present / (post_if_present.sum(axis=0, keepdims=True) + EPS)
        h_if_present = -np.sum(post_if_present * np.log(post_if_present + EPS), axis=0)
        post_if_absent = post_i[:, None] * (1.0 - p_col)
        post_if_absent = post_if_absent / (post_if_absent.sum(axis=0, keepdims=True) + EPS)
        h_if_absent = -np.sum(post_if_absent * np.log(post_if_absent + EPS), axis=0)
        expected_h = p_present_marg * h_if_present + (1 - p_present_marg) * h_if_absent
        best_local = int(np.argmin(expected_h))
        chosen_code = cands[best_local]
        posterior_v[i] = post_if_present[:, best_local]
        kept_state_v[i] = kept_state_v[i] + items_by_code_v[i][chosen_code]
        remaining_v[i].remove(chosen_code)
        code_sequence_v[i].append(chosen_code)
    if step in (5, 8):
        checkpoints_v[step] = [list(x) for x in kept_state_v]

X_5v = build_X_from_kept(checkpoints_v[5], age_verify, sex_verify)
X_8v = build_X_from_kept(checkpoints_v[8], age_verify, sex_verify)
y_pred_5v = model.predict(X_5v)
y_pred_8v = model.predict(X_8v)

sev1_mask_v = np.isin(y_verify, list(severity_1_classes))
regression_mask = sev1_mask_v & (y_pred_5v == y_verify) & (y_pred_8v != y_verify)
n_regression = int(regression_mask.sum())
print(f"عدد حالات مجموعة التراجع (صحّت عند k=5 وأخطأت عند k=8): {n_regression}")
print(f"نسبتها من حالات severity=1: {n_regression / n_sev1_verify:.4%}")

عدد حالات مجموعة التراجع (صحّت عند k=5 وأخطأت عند k=8): 115
نسبتها من حالات severity=1: 5.1020%


In [19]:
reg_indices = np.where(regression_mask)[0]
true_diseases = label_encoder.inverse_transform(y_verify[reg_indices])
pred_diseases_wrong = label_encoder.inverse_transform(y_pred_8v[reg_indices])

print("توزيع الأمراض الحقيقية (severity=1) في مجموعة التراجع:")
for disease, cnt in Counter(true_diseases).most_common():
    print(f"  {disease}: {cnt}")

print("\nتوزيع الأمراض المتوقَّعة خطأً عند k=8 (مع severity):")
pred_counts = Counter(pred_diseases_wrong)
for disease, cnt in pred_counts.most_common():
    sev = conditions[disease]["severity"]
    print(f"  {disease} (severity={sev}): {cnt}")

added_codes_counter = Counter()
for i in reg_indices:
    added_codes_counter.update(code_sequence_v[i][5:8])  # steps 6, 7, 8 = الفرق بين k=5 وk=8

print("\nالأدلة الإضافية (بين k=5 وk=8) الأكثر تكراراً في مجموعة التراجع:")
for code_id, cnt in added_codes_counter.most_common(4):
    print(f"  {code_id}: {cnt} مرة (من أصل {n_regression} حالة)")

توزيع الأمراض الحقيقية (severity=1) في مجموعة التراجع:
  Anaphylaxis: 110
  Possible NSTEMI / STEMI: 5

توزيع الأمراض المتوقَّعة خطأً عند k=8 (مع severity):
  Larygospasm (severity=1): 110
  Atrial fibrillation (severity=3): 4
  Unstable angina (severity=2): 1

الأدلة الإضافية (بين k=5 وk=8) الأكثر تكراراً في مجموعة التراجع:
  E_194: 110 مرة (من أصل 115 حالة)
  E_12: 71 مرة (من أصل 115 حالة)
  E_42: 57 مرة (من أصل 115 حالة)


<div dir="rtl">

**التفسير**: 110 من 115 حالة (95.7%) هي التباس **Anaphylaxis ↔ Larygospasm — كلاهما severity=1** (ليس
تراجعاً في الخطورة). الأدلة الثلاثة الأكثر ارتباطاً بهذا الالتباس كلها متعلقة بالحساسية/انسداد مجرى الهواء
العلوي (من `release_evidences.json`):

- `E_194` — "Have you noticed a high pitched sound when breathing in?" (صرير تنفسي)
- `E_12` — "Do you have a known severe food allergy?"
- `E_42` — "Have you been in contact with or ate something that you have an allergy to?"

هذه الأعراض مشتركة تماماً بين المرضين (كلاهما انسداد مجرى هوائي علوي حاد، غالباً بمحفِّز تحسسي) — كشفها يرفع
الثقة بـ"طارئ تنفسي تحسسي" عموماً دون أن يميّز جيداً بين الاثنين تحديداً، فيميل `EIG` أحياناً للتشخيص الخطأ
ضمن هذا الزوج شديد التشابه. الـ5 حالات المتبقية (Possible NSTEMI/STEMI → Atrial fibrillation/Unstable angina)
هي التراجع الحقيقي الوحيد في تقدير الخطورة ضمن مجموعة التراجع كاملة، ويتفق ذلك تماماً مع `undertriage_rate`
المنخفض جداً عند k=8 (0.29%) — 5 حالات من 30,000 هي بالضبط بهذا الحجم.

</div>

<div dir="rtl">

## تفسير: `expected_information_gain` يتفوَّق بوضوح — خصوصاً على severity=1

**Accuracy**: `EIG` هو الأفضل عند k=3, 5, 8 (74.0%، 89.4%، 94.1% على التوالي) — يتفوَّق على `random`
و`greedy_entropy` معاً بفارق واضح. عند k=12 يتقارب مع `random` (97.9% مقابل 98.1%، فرق ضئيل داخل حدود
الضجيج)، وكلاهما بعيد جداً عن `greedy_entropy` (93.2%).

**severity=1 — الأهم سريرياً**: `EIG` هو **الأفضل عند كل قيم k بلا استثناء**، وغالباً بفارق كبير:

| k | random | greedy_entropy | expected_information_gain |
|---|---|---|---|
| 3 | 60.3% | 50.7% | **20.8%** |
| 5 | 40.6% | 46.5% | **5.1%** |
| 8 | 17.8% | 39.3% | **9.6%** |
| 12 | 3.9% | 35.0% | **2.1%** |

`EIG` يتفوَّق حتى على `random` عند k=12 (2.1% مقابل 3.9%) رغم استهلاك أدلة أقل قليلاً في المتوسط (15.53 مقابل
15.39 — متقاربان). هذا **متسق مع الحدس**: `EIG` يبني معتقداً بايزياً صريحاً عبر الأمراض الـ49 (لا الاثنين
الأكثر شيوعاً فقط)، فيبقى حساساً لاحتمال مرض نادر/خطير طالما لم يُستبعَد فعلياً بدليل حقيقي — بعكس
`greedy_entropy` الذي يتسرَّع بتفضيل أي دليل يرفع ثقة النموذج **بأي** استنتاج، حتى لو كان استنتاجاً شائعاً
خاطئاً يُبعِد عن التشخيص الصحيح النادر.

**لماذا `EIG` أفضل من `greedy_entropy` رغم اعتماد الأول على نموذج Bayes "ساذج" أبسط من `LogisticRegression`
المُستخدَم في الثاني؟** لأن المعيار نفسه مختلف جوهرياً: `EIG` يقيس **قيمة السؤال قبل معرفة الإجابة** (لا
يستطيع "الغش")، بينما `greedy_entropy` يقيس **الثقة بعد معرفة الإجابة الحقيقية فعلياً** — وهذا الأخير يُكافئ
الأسئلة التي تُثبِّت أي استنتاج (صحيحاً أو خاطئاً)، لا الأسئلة الأكثر تمييزاً فعلياً بين الاحتمالات المتنافسة.
جودة النموذج الأساسي (Bayes ساذج مقابل Logistic Regression) أقل أهمية هنا من *شكل معيار الاختيار نفسه*.

**الخلاصة العملية**: `expected_information_gain` مرشَّح جيد لمنطق اختيار الأسئلة في مقابلة سريرية تفاعلية —
أداء أفضل من `random` في أغلب النطاق، وأفضل بكثير من `greedy_entropy` في كل النطاق، وتحديداً في الحالات
الطارئة. يبقى قابلاً للتحسين (تبسيط "موجود/غائب" فقط لكل كود يتجاهل قيمة الأدلة C/M المحدَّدة)، لكنه أساس أمتن
بكثير من الاختيار الجشع بالإنتروبيا الفعلية.

</div>

<div dir="rtl">

## ملخص وخارج النطاق

- **تحقيق الاختلاف مع 03**: أُثبت تجريبياً أن الفارق بين أرقام `random` في هذا النوتبوك (تشغيله الأول)
  و`03_train.ipynb` ناتج عن أثرين صغيرين مجتمعين (تباين معاينة عيّنة أصغر + ترتيب استهلاك RNG تسلسلي مختلف)،
  لا خطأ في المنهجية. الإصلاح: بذرة عشوائية مستقلة لكل صف (`(RANDOM_STATE, row_position)`)، مطبَّقة على
  `random` في هذا النوتبوك من الآن فصاعداً.
- **`greedy_entropy` محفوظ كاملاً دون حذف** (نفس الكود، نفس النتائج الحتمية) — نتيجة سلبية موثَّقة سابقاً
  ومؤكَّدة هنا مجدداً.
- **`expected_information_gain` جديد**: جدول `P(code|disease)` من `train` الكامل (Bayes ساذج، تبسيط
  موجود/غائب فقط)، اختيار بمعيار التوقع لا الجواب الفعلي، تشخيص نهائي دائماً بالنموذج المدرَّب. **الأفضل بين
  الثلاثة على severity=1 عند كل قيم k**، والأفضل إجمالاً عند k=3,5,8.
- **`undertriage_rate` و`same_severity_confusion` مضافان لكل الجداول** (لا حذف للمقاييس القديمة). النتيجة
  الحاسمة: شذوذ k=8 في "نسبة الخطأ severity=1" **يختفي تماماً** تحت `undertriage_rate` (ينخفض رتيباً)، لأنه
  كان بالكامل تقريباً (95.7%) التباساً تشخيصياً بين مرضين بنفس درجة الخطورة (Anaphylaxis/Larygospasm)، لا
  تراجعاً حقيقياً في تقدير الخطورة — مؤكَّد بتحليل مجموعة التراجع صريحاً (الأدلة الثلاثة المسبِّبة، وتوزيع
  الأمراض).
- نفس عيّنة `validate` الفرعية (10,000 صف، `random_state=42`) لكل الطرق الثلاث للجدول الرئيسي — مقارنة عادلة
  (تحليل مجموعة التراجع استخدم عيّنة تحقق أكبر، 30,000، لثبات إحصائي أعلى، مذكور صراحة). `test` لم يُلمس. لا
  XGBoost.

</div>